### 🔗 Links and Resources
| Method / Config | Description |
|-----------------|-------------|
| [DataFrame.join()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html) | Joins two DataFrames on a condition or column name |
| [DataFrame.hint()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.hint.html) | Applies a strategy hint — use `"merge"` to force Sort Merge Join |
| [DataFrame.explain()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html) | Prints the physical plan — look for `SortMergeJoin` and `Exchange` |
| [DataFrame.repartition()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.repartition.html) | Reshuffles data into N partitions (hash-based) |
| [DataFrame.sortWithinPartitions()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.sortWithinPartitions.html) | Sorts rows within each partition without global sort |
| `spark.sql.autoBroadcastJoinThreshold` | Set to `-1` to disable auto-broadcast and expose SMJ plans |
| `spark.sql.join.preferSortMergeJoin` | When `true` (default), Spark prefers SMJ over Shuffled Hash Join |
| `spark.sql.shuffle.partitions` | Number of partitions after a shuffle (default 200) |
| `spark.sql.adaptive.enabled` | Enables Adaptive Query Execution (AQE) — can convert SMJ to BHJ at runtime |
| `spark.sql.adaptive.skewJoin.enabled` | AQE sub-feature that splits skewed partitions in an SMJ |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

### 📌 What Is a Sort Merge Join?

**Sort Merge Join (SMJ)** is Spark's default join strategy for **large-to-large table joins** where neither side fits in memory for broadcasting.

It works in three phases:

```
Phase 1 — SHUFFLE
  Both tables are hash-partitioned by the join key.
  Rows with the same key end up in the same partition on the same executor.
  This is the most expensive phase (network I/O).

  [Partition 0: keys 0-99 ]   [Partition 0: keys 0-99 ]
  [Partition 1: keys 100-199] [Partition 1: keys 100-199]  ... etc.

Phase 2 — SORT
  Within each partition, rows are sorted by the join key.
  After sorting, matching rows are adjacent — no hashing needed.

Phase 3 — MERGE
  A linear scan through both sorted partitions simultaneously.
  Like merging two sorted arrays — O(n) per partition.
  Matching rows are emitted; one or both pointers advance.
```

#### Why two shuffles?

Each side is an independent DataFrame — they have no guarantee that rows with
the same key land on the same executor. The shuffle co-locates them.
Both sides are reshuffled with the **same partitioner** so partition _i_ of
table A only needs to be compared with partition _i_ of table B.

#### When does Spark choose SMJ?

| Condition | Strategy Spark picks |
|-----------|---------------------|
| Small table ≤ `autoBroadcastJoinThreshold` (default 10 MB) | Broadcast Hash Join |
| Both tables large, `preferSortMergeJoin = true` (default) | **Sort Merge Join** |
| Both tables large, `preferSortMergeJoin = false`, build side fits in memory | Shuffled Hash Join |
| AQE enabled, runtime stats show one side is now small | AQE converts to Broadcast Hash Join at runtime |

#### Plan nodes to look for

| Node | Meaning |
|------|---------|
| `SortMergeJoin` / `PhotonSortMergeJoin` | SMJ is executing ✅ |
| `Exchange hashpartitioning(...)` | Shuffle phase — rows are repartitioned by join key |
| `Sort [key ASC]` | Sort phase — partition rows sorted before merge |
| `AdaptiveSparkPlan` | AQE is active; final strategy may differ from initial plan |

### 📌 Creating Two Large DataFrames

Both tables must be **above the broadcast threshold** to trigger SMJ.
We use `range()` to generate large in-memory datasets, then add a join key column.

In [ ]:
from pyspark.sql import functions as F

# ── Left table: 5 million rows simulating an orders fact table ────────────────
df_orders = (
    spark.range(5_000_000)          # id: 0 .. 4_999_999
    .withColumnRenamed("id", "order_id")
    .withColumn("customer_id", (F.col("order_id") % 1_000_000).cast("int"))  # FK
    .withColumn("amount",      (F.rand(seed=42) * 500 + 10).cast("decimal(10,2)"))
    .withColumn("status",      F.when(F.col("order_id") % 3 == 0, "shipped")
                                .when(F.col("order_id") % 3 == 1, "pending")
                                .otherwise("cancelled"))
)

# ── Right table: 1 million rows simulating a customers dimension table ────────
df_customers = (
    spark.range(1_000_000)          # id: 0 .. 999_999
    .withColumnRenamed("id", "customer_id")
    .withColumn("country",  F.when(F.col("customer_id") % 4 == 0, "US")
                             .when(F.col("customer_id") % 4 == 1, "UK")
                             .when(F.col("customer_id") % 4 == 2, "DE")
                             .otherwise("FR"))
    .withColumn("tier",     F.when(F.col("customer_id") % 10 == 0, "gold")
                             .when(F.col("customer_id") % 10 < 4,  "silver")
                             .otherwise("bronze"))
)

print(f"orders     row count : {df_orders.count():>12,}")
print(f"customers  row count : {df_customers.count():>12,}")
print()
print("orders schema:")
df_orders.printSchema()
print("customers schema:")
df_customers.printSchema()

### 📌 Reading Relevant Configuration Values

In [ ]:
def get_conf(key, default="(not set)"):
    try:
        return spark.conf.get(key)
    except Exception:
        return default

configs = [
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.join.preferSortMergeJoin",
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
]

print(f"{'Config key':<55} {'Value'}")
print("-" * 75)
for key in configs:
    print(f"{key:<55} {get_conf(key)}")

### 📌 Triggering Sort Merge Join

With auto-broadcast disabled, neither table can be broadcast — Spark falls back
to Sort Merge Join (because `preferSortMergeJoin = true` by default).

We disable auto-broadcast here to make the SMJ visible in the plan even on
Databricks serverless, which can be aggressive with broadcast thresholds.

In [ ]:
# Disable auto-broadcast so Spark cannot choose Broadcast Hash Join.
# This is the standard way to observe SMJ in the physical plan.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

df_smj = df_orders.join(df_customers, on="customer_id", how="inner")

# Show a sample of the result
df_smj.show(10, truncate=False)

### 📌 Reading the Physical Plan

Look for these nodes in order (bottom-up — Spark plans read leaves → root):

```
LocalTableScan / Scan                  ← leaf: read the raw data
  Exchange hashpartitioning(key, N)    ← Phase 1: shuffle by join key
    Sort [key ASC NULLS FIRST]         ← Phase 2: sort each partition
      SortMergeJoin [key], [key], ...  ← Phase 3: linear merge
```

On **Databricks with Photon** the nodes are prefixed with `Photon`:
`PhotonSortMergeJoin`, `PhotonShuffleExchangeSink`, etc.

In [ ]:
print("=" * 70)
print("Simple plan (physical only)")
print("=" * 70)
df_smj.explain(mode="simple")

print("=" * 70)
print("Formatted plan (easiest to read)")
print("=" * 70)
df_smj.explain(mode="formatted")

### 📌 Forcing SMJ with `hint("merge")`

`hint("merge")` is the explicit way to request Sort Merge Join regardless of table
sizes or configuration. It overrides AQE runtime decisions and auto-broadcast.

Useful when:
- You know the data will be large at runtime even if statistics suggest otherwise.
- AQE is converting your expected SMJ to a broadcast join and you want to prevent it.
- You need deterministic, reproducible plans in tests.

In [ ]:
# Re-enable auto-broadcast to show that hint("merge") overrides it
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))  # 10 MB

# Without hint — Spark may choose Broadcast Hash Join if df_customers fits threshold
df_no_hint = df_orders.join(df_customers, on="customer_id", how="inner")
print("Plan WITHOUT hint (auto-broadcast enabled):")
df_no_hint.explain(mode="simple")

# With hint("merge") on either side — forces SMJ regardless of thresholds
df_forced_smj = df_orders.join(
    df_customers.hint("merge"),
    on="customer_id",
    how="inner"
)
print("Plan WITH hint('merge') — SMJ is forced:")
df_forced_smj.explain(mode="simple")

### 📌 `spark.sql.shuffle.partitions` — Controlling Parallelism

The shuffle in Phase 1 creates **N partitions** where N = `spark.sql.shuffle.partitions`
(default **200**).

| Scenario | Recommendation |
|----------|---------------|
| Small cluster / small data | Reduce below 200 — fewer, fuller partitions are faster |
| Large cluster / large data | Increase above 200 — more parallelism, less spill |
| AQE enabled | AQE coalesces empty/tiny shuffle partitions automatically — less manual tuning needed |

**Rule of thumb**: target partition sizes of **128 MB – 256 MB** after the shuffle.

```python
# Too few: large partitions spill to disk
spark.conf.set("spark.sql.shuffle.partitions", "50")

# Too many: task scheduling overhead dominates
spark.conf.set("spark.sql.shuffle.partitions", "10000")

# AQE auto-coalescing (recommended): let Spark decide at runtime
spark.conf.set("spark.sql.adaptive.enabled", "true")   # already default on Databricks
```

In [ ]:
# Disable auto-broadcast to keep SMJ in the plan
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

for n_parts in [10, 200, 400]:
    spark.conf.set("spark.sql.shuffle.partitions", str(n_parts))
    df_test = df_orders.join(df_customers, on="customer_id", how="inner")
    # Count output partitions of the join result
    result_parts = df_test.rdd.getNumPartitions()
    print(f"shuffle.partitions={n_parts:>4}  →  result partitions: {result_parts}")

# Reset to default
spark.conf.set("spark.sql.shuffle.partitions", "200")
print("\nReset shuffle.partitions to 200")

### 📌 AQE and Sort Merge Join

**Adaptive Query Execution (AQE)** (enabled by default in Spark 3.x and Databricks)
can change the join strategy **at runtime** after it has collected actual row/byte
statistics from the shuffle.

#### Three AQE features that directly affect SMJ

| AQE Feature | Config | What it does |
|-------------|--------|--------------|
| **Dynamic Join Switch** | `spark.sql.adaptive.enabled` | Converts a planned SMJ to Broadcast Hash Join if the runtime size of one side falls below the broadcast threshold |
| **Coalesce Shuffle Partitions** | `spark.sql.adaptive.coalescePartitions.enabled` | Merges small post-shuffle partitions to avoid thousands of tiny tasks |
| **Skew Join Handling** | `spark.sql.adaptive.skewJoin.enabled` | Detects and splits **skewed partitions** (one key has far more rows than others) by sub-partitioning the skewed side |

#### What is data skew in an SMJ?

```
Normal distribution:          Skewed distribution:
  partition 0: 1 000 rows       partition 0:   1 000 rows
  partition 1: 1 000 rows       partition 1: 500 000 rows  ← hot partition
  partition 2: 1 000 rows       partition 2:     500 rows

Skew causes one executor to run for minutes while others finish in seconds.
AQE splits the hot partition into sub-partitions and replicates the matching
rows from the other side to each sub-partition.
```

A partition is considered skewed when it is **both**:
- larger than `skewedPartitionThresholdInBytes` (default **256 MB**), **and**
- larger than `skewedPartitionFactor` × median partition size (default **5×**)

In [ ]:
# ── Demonstrate AQE dynamic join switch ──────────────────────────────────────
# With AQE enabled and auto-broadcast on, Spark may start planning an SMJ
# but switch to BHJ once it knows the actual size of the smaller side.

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))  # 10 MB
spark.conf.set("spark.sql.shuffle.partitions", "200")

df_aqe = df_orders.join(df_customers, on="customer_id", how="inner")

print("Plan with AQE enabled — note 'AdaptiveSparkPlan isFinalPlan=false' in initial plan.")
print("After execution, isFinalPlan=true and the strategy may have changed.")
print()
df_aqe.explain(mode="formatted")

### 📌 Creating a Skewed Dataset to Observe Skew Handling

In [ ]:
from pyspark.sql.types import StructType, StructField, LongType, StringType

# ── Build a skewed left table ─────────────────────────────────────────────────
# customer_id=0 appears 80 % of the time — a classic hot key.
df_skewed_orders = (
    spark.range(2_000_000)
    .withColumnRenamed("id", "order_id")
    .withColumn(
        "customer_id",
        # 80 % of rows → customer_id = 0  (hot key)
        F.when(F.rand(seed=7) < 0.8, F.lit(0))
         .otherwise((F.col("order_id") % 999 + 1).cast("long"))
    )
    .withColumn("amount", (F.rand(seed=99) * 200).cast("decimal(10,2)"))
)

# ── Right table: 1 000 customers ──────────────────────────────────────────────
df_small_customers = (
    spark.range(1000)
    .withColumnRenamed("id", "customer_id")
    .withColumn("tier", F.when(F.col("customer_id") == 0, "vip").otherwise("standard"))
)

# Show skew distribution — customer_id=0 should dominate
print("Top 5 customer_id frequencies in skewed_orders:")
(
    df_skewed_orders
    .groupBy("customer_id")
    .count()
    .orderBy(F.col("count").desc())
    .show(5)
)

In [ ]:
# ── Compare plans: skew handling OFF vs ON ────────────────────────────────────
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force SMJ

# OFF: all rows for customer_id=0 pile into one partition → slow executor
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
df_skew_off = df_skewed_orders.join(df_small_customers, on="customer_id", how="inner")
print("Plan with skewJoin.enabled = FALSE:")
df_skew_off.explain(mode="simple")

# ON: AQE detects the hot partition and splits it into sub-partitions
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
df_skew_on = df_skewed_orders.join(df_small_customers, on="customer_id", how="inner")
print("Plan with skewJoin.enabled = TRUE:")
df_skew_on.explain(mode="simple")

# Reset
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

### 📌 Pre-Partitioned Data — Eliminating the Shuffle

If both DataFrames are **already partitioned by the join key** with the same number
of partitions, Spark can skip the shuffle phase entirely — the most expensive step.

This is called a **Sort Merge Join without Exchange** and is the fastest possible
large-table join.

In practice this happens when:
- Both tables are read from **bucketed Hive/Delta tables** on the same key and bucket count.
- You explicitly `repartition()` both DataFrames on the join key before joining.

> **Note:** On Databricks with AQE enabled, explicitly `repartition()`-ing in Python
> code may still show an `Exchange` in the initial plan but AQE can eliminate it
> at runtime. Use bucketed Delta tables for a reliable shuffle-free SMJ in production.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # keep SMJ visible
N = 200  # must match on both sides

# ── Without pre-partitioning: two Exchange nodes in the plan ──────────────────
df_plain = df_orders.join(df_customers, on="customer_id", how="inner")
print("Plan WITHOUT pre-partitioning (expect two Exchange nodes):")
df_plain.explain(mode="simple")

# ── With pre-partitioning: Spark MAY eliminate one or both Exchange nodes ─────
df_orders_pre     = df_orders.repartition(N, "customer_id")
df_customers_pre  = df_customers.repartition(N, "customer_id")
df_pre = df_orders_pre.join(df_customers_pre, on="customer_id", how="inner")
print("Plan WITH pre-partitioning (Exchange nodes may be reduced):")
df_pre.explain(mode="simple")

# Reset
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

### 📌 SMJ vs Other Join Strategies — Decision Guide

| Strategy | Best when | Cost | Plan node |
|----------|-----------|------|-----------|
| **Broadcast Hash Join** | One side ≤ broadcast threshold | Cheapest (no shuffle, no sort) | `BroadcastHashJoin` |
| **Sort Merge Join** | Both sides large, equi-join on sortable key | Shuffle + Sort + Merge | `SortMergeJoin` |
| **Shuffled Hash Join** | Both sides large, build side fits in executor memory | Shuffle + Hash (no sort) | `ShuffledHashJoin` |
| **Broadcast Nested Loop Join** | Non-equi join (e.g. `CROSS JOIN`, range join) | Most expensive — cartesian product | `BroadcastNestedLoopJoin` |
| **Cartesian Product Join** | Explicit `CROSS JOIN` with no key | Catastrophically expensive | `CartesianProduct` |

#### Prefer SMJ when
- Both tables are **hundreds of MB to TB** in size.
- The join key has **good cardinality** (no severe skew) — or AQE skew handling is on.
- Data is already **sorted or bucketed** on the join key (shuffle eliminated).
- You need **outer joins** — SMJ supports all join types; Shuffled Hash Join has limitations.

#### Avoid SMJ when
- One side is small enough to broadcast — BHJ is always faster.
- The join key is **non-sortable** (e.g., arrays, maps) — SMJ requires ordering.
- The join condition is **non-equi** (e.g., `a.date BETWEEN b.start AND b.end`) — use BNL or range-join optimization.

### 📌 Configuration Cheat Sheet

```python
# Disable auto-broadcast (expose SMJ in plans during development)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# Force SMJ preference over Shuffled Hash Join (default true)
spark.conf.set("spark.sql.join.preferSortMergeJoin", "true")

# Tune shuffle parallelism (default 200; target 128–256 MB per partition)
spark.conf.set("spark.sql.shuffle.partitions", "400")

# Enable AQE (default true in Spark 3.x / Databricks)
spark.conf.set("spark.sql.adaptive.enabled", "true")

# Enable AQE coalesce — merges tiny post-shuffle partitions (default true)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Enable AQE skew join — split hot partitions (default true)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# Skew thresholds (partition must exceed BOTH to be considered skewed)
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", str(256 * 1024 * 1024))
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
```